# optuna

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/"Colab Notebooks"

/content/drive/MyDrive/Colab Notebooks


In [ ]:


# Install Gymnasium + MuJoCo support
!pip install "gymnasium[mujoco]==0.29.1" stable-baselines3==2.2.1
!pip install imitation optuna


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.7/181.7 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 22.1 MB/s eta 0:00:00
  Attempting uninstall: gymnasium
    Found existing installation: gymnasium 1.2.3
    Uninstalling gymnasium-1.2.3:
      Successfully uninstalled gymnasium-1.2.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.5/216.5 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 33.7 MB/s eta 0:00:00
 

In [ ]:
optuna.visualization.plot_param_importances(study)

In [ ]:
import os
import time
import shutil
from dataclasses import dataclass

import gymnasium as gym
import numpy as np

import optuna
from optuna.pruners import MedianPruner

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, EvalCallback
from stable_baselines3.common.vec_env import VecMonitor

def objective(trial):

    lr = trial.suggest_loguniform("learning_rate", 1e-5, 3e-3)
    gamma = trial.suggest_float("gamma", 0.95, 0.999)
    n_steps = trial.suggest_categorical("n_steps", [1024, 2048, 4096])

    env = gym.make("Ant-v4")

    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=lr,
        gamma=gamma,
        n_steps=n_steps,
        verbose=0,
    )

    model.learn(total_timesteps=20000)

    # Evaluate
    obs, _ = env.reset()
    total_reward = 0

    for _ in range(1000):
        action, _ = model.predict(obs)
        obs, reward, done, trunc, _ = env.step(action)
        total_reward += reward
        if done or trunc:
            break

    env.close()

    return total_reward


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print(study.best_params)


[I 2026-02-11 07:42:11,163] A new study created in memory with name: no-name-bba2cc16-dfb3-4fb3-9885-1ce2faf16d0a
/tmp/ipython-input-283621145.py:19: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("learning_rate", 1e-5, 3e-3)
[I 2026-02-11 07:42:59,944] Trial 0 finished with value: -14.336484437243202 and parameters: {'learning_rate': 0.00033504823466318754, 'gamma': 0.956376678278462, 'n_steps': 1024}. Best is trial 0 with value: -14.336484437243202.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/tmp/ipython-input-283621145.py:1

{'learning_rate': 0.0006014791628921789, 'gamma': 0.9660823983708584, 'n_steps': 4096}


In [ ]:
optuna.visualization.plot_optimization_history(study)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



In [ ]:
optuna.visualization.plot_param_importances(study)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



In [ ]:
# ============================================================
# ONE-CELL: Optuna + SB3 PPO (MuJoCo) with pruning, TensorBoard,
#           checkpoints every 500k, best model save, study resume
# ============================================================
!pip -q install -U "stable-baselines3[extra]" optuna tensorboard gymnasium mujoco==3.* shimmy

import os, re, glob, time, json, math
from dataclasses import dataclass
from typing import Optional, Dict, Any

import numpy as np
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv, VecMonitor, VecNormalize
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback, CallbackList, CheckpointCallback
from stable_baselines3.common.evaluation import evaluate_policy

# -----------------------------
# Config
# -----------------------------
ENV_ID = "Ant-v4"  # try: "Hopper-v4", "Walker2d-v4", "HalfCheetah-v4"
TOTAL_TIMESTEPS = 3_000_000
N_ENVS = 16
EVAL_FREQ = 50_000
N_EVAL_EPISODES = 5
CHECKPOINT_FREQ = 500_000
SEED = 0

ROOT = "/content/ppo_optuna"
STUDY_DB = os.path.join(ROOT, "optuna_study.sqlite3")
STUDY_NAME = f"ppo_{ENV_ID.lower().replace('-', '_')}"
TB_ROOT = os.path.join(ROOT, "tb")
TRIAL_ROOT = os.path.join(ROOT, "trials")
BEST_ROOT = os.path.join(ROOT, "best")

os.makedirs(ROOT, exist_ok=True)
os.makedirs(TB_ROOT, exist_ok=True)
os.makedirs(TRIAL_ROOT, exist_ok=True)
os.makedirs(BEST_ROOT, exist_ok=True)

# -----------------------------
# Env helpers
# -----------------------------
def make_env(env_id: str, rank: int, seed: int, log_dir: str):
    def _init():
        env = gym.make(env_id)
        env = Monitor(env, filename=os.path.join(log_dir, f"monitor_rank{rank}.csv"))
        env.reset(seed=seed + rank)
        return env
    return _init

def build_vec_env(env_id: str, n_envs: int, seed: int, log_dir: str):
    if n_envs == 1:
        venv = DummyVecEnv([make_env(env_id, 0, seed, log_dir)])
    else:
        venv = SubprocVecEnv([make_env(env_id, i, seed, log_dir) for i in range(n_envs)])
    venv = VecMonitor(venv)
    # VecNormalize helps stability; save it in checkpoints
    venv = VecNormalize(venv, norm_obs=True, norm_reward=True, clip_obs=10.0)
    return venv

# -----------------------------
# Callback: reports to Optuna and prunes
# -----------------------------
class OptunaEvalCallback(BaseCallback):
    def __init__(self, trial: optuna.Trial, eval_env, eval_freq: int, n_eval_episodes: int, trial_dir: str):
        super().__init__()
        self.trial = trial
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.n_eval_episodes = n_eval_episodes
        self.trial_dir = trial_dir
        self.best_mean = -np.inf
        self.log_path = os.path.join(trial_dir, "eval_history.jsonl")
        self.last_eval_step = 0

    def _on_step(self) -> bool:
        if (self.num_timesteps - self.last_eval_step) < self.eval_freq:
            return True

        self.last_eval_step = self.num_timesteps
        mean_r, std_r = evaluate_policy(
            self.model, self.eval_env, n_eval_episodes=self.n_eval_episodes, deterministic=True
        )

        # report intermediate value to Optuna
        self.trial.report(mean_r, step=self.num_timesteps)

        # persist eval history
        with open(self.log_path, "a") as f:
            f.write(json.dumps({
                "timestep": int(self.num_timesteps),
                "mean_reward": float(mean_r),
                "std_reward": float(std_r),
            }) + "\n")

        # track best mean reward for this trial
        if mean_r > self.best_mean:
            self.best_mean = mean_r
            # save best model for this trial
            self.model.save(os.path.join(self.trial_dir, "model_best"))
            # also save VecNormalize stats
            try:
                self.eval_env.save(os.path.join(self.trial_dir, "vecnormalize.pkl"))
            except Exception:
                pass

        # prune if needed
        if self.trial.should_prune():
            raise optuna.TrialPruned(f"Pruned at step={self.num_timesteps}, mean_r={mean_r:.2f}")

        return True

# -----------------------------
# Checkpoint resume helper
# -----------------------------
def latest_checkpoint(trial_dir: str) -> Optional[str]:
    ckpts = glob.glob(os.path.join(trial_dir, "checkpoints", "model_*_steps.zip"))
    if not ckpts:
        return None
    def steps(path):
        m = re.search(r"model_(\d+)_steps\.zip$", os.path.basename(path))
        return int(m.group(1)) if m else -1
    ckpts.sort(key=steps)
    return ckpts[-1]

# -----------------------------
# Optuna objective
# -----------------------------
def objective(trial: optuna.Trial) -> float:
    # trial dirs
    trial_dir = os.path.join(TRIAL_ROOT, f"trial_{trial.number:04d}")
    os.makedirs(trial_dir, exist_ok=True)
    ckpt_dir = os.path.join(trial_dir, "checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)

    # hyperparams to tune
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 3e-3, log=True)
    n_steps = trial.suggest_categorical("n_steps", [128, 256, 512, 1024, 2048])
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256, 512])
    gamma = trial.suggest_float("gamma", 0.90, 0.9999, log=True)
    gae_lambda = trial.suggest_float("gae_lambda", 0.80, 0.99)
    clip_range = trial.suggest_float("clip_range", 0.1, 0.3)
    ent_coef = trial.suggest_float("ent_coef", 1e-8, 1e-2, log=True)
    vf_coef = trial.suggest_float("vf_coef", 0.1, 1.0)
    max_grad_norm = trial.suggest_float("max_grad_norm", 0.3, 1.0)

    # policy net
    net_arch = trial.suggest_categorical("net_arch", ["small", "medium", "large"])
    if net_arch == "small":
        policy_kwargs = dict(net_arch=dict(pi=[64, 64], vf=[64, 64]))
    elif net_arch == "medium":
        policy_kwargs = dict(net_arch=dict(pi=[128, 128], vf=[128, 128]))
    else:
        policy_kwargs = dict(net_arch=dict(pi=[256, 256], vf=[256, 256]))

    # tensorboard log dir for this trial
    tb_log = os.path.join(TB_ROOT, f"trial_{trial.number:04d}")

    # build train + eval envs
    env_log_dir = os.path.join(trial_dir, "env_logs")
    os.makedirs(env_log_dir, exist_ok=True)
    train_env = build_vec_env(ENV_ID, N_ENVS, SEED + 1000 * trial.number, env_log_dir)
    eval_env  = build_vec_env(ENV_ID, 1,     SEED + 2000 * trial.number, env_log_dir)

    # resume-from-checkpoint inside the trial (optional)
    ckpt = latest_checkpoint(trial_dir)
    if ckpt is not None:
        # load model and attach env
        model = PPO.load(ckpt, env=train_env, device="auto", print_system_info=False)
        # try to restore VecNormalize stats
        vn_path = os.path.join(trial_dir, "vecnormalize.pkl")
        if os.path.exists(vn_path):
            train_env = VecNormalize.load(vn_path, train_env)
            eval_env = VecNormalize.load(vn_path, eval_env)
            model.set_env(train_env)
        already = int(re.search(r"model_(\d+)_steps\.zip$", os.path.basename(ckpt)).group(1))
    else:
        model = PPO(
            "MlpPolicy",
            train_env,
            learning_rate=learning_rate,
            n_steps=n_steps,
            batch_size=batch_size,
            gamma=gamma,
            gae_lambda=gae_lambda,
            clip_range=clip_range,
            ent_coef=ent_coef,
            vf_coef=vf_coef,
            max_grad_norm=max_grad_norm,
            policy_kwargs=policy_kwargs,
            verbose=0,
            tensorboard_log=TB_ROOT,
            seed=SEED + trial.number,
            device="auto",
        )
        already = 0

    # callbacks: periodic checkpoints + optuna eval/prune
    checkpoint_cb = CheckpointCallback(
        save_freq=CHECKPOINT_FREQ // N_ENVS,   # SB3 counts "calls", so divide by num envs
        save_path=ckpt_dir,
        name_prefix="model",
        save_replay_buffer=False,
        save_vecnormalize=True,
    )

    optuna_cb = OptunaEvalCallback(
        trial=trial,
        eval_env=eval_env,
        eval_freq=EVAL_FREQ,
        n_eval_episodes=N_EVAL_EPISODES,
        trial_dir=trial_dir,
    )

    callbacks = CallbackList([checkpoint_cb, optuna_cb])

    # train
    remaining = max(0, TOTAL_TIMESTEPS - already)
    if remaining == 0:
        # if already done, evaluate and return
        mean_r, _ = evaluate_policy(model, eval_env, n_eval_episodes=N_EVAL_EPISODES, deterministic=True)
        return float(mean_r)

    model.learn(
        total_timesteps=remaining,
        reset_num_timesteps=False,
        tb_log_name=f"trial_{trial.number:04d}",
        callback=callbacks,
        progress_bar=False,
    )

    # final eval
    mean_r, _ = evaluate_policy(model, eval_env, n_eval_episodes=N_EVAL_EPISODES, deterministic=True)

    # save final
    model.save(os.path.join(trial_dir, "model_final"))
    try:
        eval_env.save(os.path.join(trial_dir, "vecnormalize.pkl"))
    except Exception:
        pass

    return float(mean_r)

# -----------------------------
# Create / resume study
# -----------------------------
storage = f"sqlite:///{STUDY_DB}"
sampler = TPESampler(seed=SEED)
pruner = MedianPruner(n_startup_trials=5, n_warmup_steps=2, interval_steps=EVAL_FREQ)

study = optuna.create_study(
    study_name=STUDY_NAME,
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
    storage=storage,
    load_if_exists=True,
)

print("Study:", study.study_name)
print("Storage:", storage)
print("Trials so far:", len(study.trials))

# -----------------------------
# Run optimization
# -----------------------------
N_TRIALS = 10  # change
try:
    study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True, show_progress_bar=False)
finally:
    print("\nBest value:", study.best_value if study.best_trial else None)
    if study.best_trial:
        print("Best params:", study.best_trial.params)

# -----------------------------
# Save best model artifact
# -----------------------------
if study.best_trial:
    best_trial_dir = os.path.join(TRIAL_ROOT, f"trial_{study.best_trial.number:04d}")
    # Prefer model_best if present else model_final
    best_model_path = os.path.join(best_trial_dir, "model_best.zip")
    if not os.path.exists(best_model_path):
        best_model_path = os.path.join(best_trial_dir, "model_final.zip")
    if os.path.exists(best_model_path):
        os.makedirs(BEST_ROOT, exist_ok=True)
        dst = os.path.join(BEST_ROOT, f"{ENV_ID}_best_trial_{study.best_trial.number:04d}.zip")
        !cp -f "{best_model_path}" "{dst}"
        print("Saved best model to:", dst)
    else:
        print("Could not find best model file in:", best_trial_dir)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 122.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 16.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires tensorboard~=2.19.0, but you have tensorboard 2.20.0 which is incompatible.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
[I 2026-02-11 08:06:19,542] A new study created in RDB with name: ppo_ant_v4


Study: ppo_ant_v4
Storage: sqlite:////content/ppo_optuna/optuna_study.sqlite3
Trials so far: 0


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/vec_monitor.py:44: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment Ant-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(
[I 2026-02-11 08:37:15,986] Trial 0 finished with value: 3290.83203125 and parameters: {'learning_rate': 0.0002288113668475521, 'n_steps': 128, 'batch_size': 256, 'gamma': 0.9782176048511658, 'gae_lambda': 0.9004900347530519, 'clip_range': 0.21360891221878647, 'ent_coef': 0.0035775015430827006, 'vf_coef': 0.16393245237809825, 'max_grad_norm': 0.3609905097910785, 'net_arch': 'medium'}. Best is trial 0 with value: 3290.83203125.
/usr/local/lib/python3.12/dist-packages/jupyter_client/


Best value: 3290.83203125
Best params: {'learning_rate': 0.0002288113668475521, 'n_steps': 128, 'batch_size': 256, 'gamma': 0.9782176048511658, 'gae_lambda': 0.9004900347530519, 'clip_range': 0.21360891221878647, 'ent_coef': 0.0035775015430827006, 'vf_coef': 0.16393245237809825, 'max_grad_norm': 0.3609905097910785, 'net_arch': 'medium'}
Saved best model to: /content/ppo_optuna/best/Ant-v4_best_trial_0000.zip


In [ ]:
!pip -q install -U plotly optuna

In [ ]:
import optuna
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_slice,
    plot_parallel_coordinate,
)

# --- point to your study DB ---
ROOT = "/content/ppo_optuna"
STUDY_DB = f"sqlite:///{ROOT}/optuna_study.sqlite3"
STUDY_NAME = "ppo_ant_v4"   # <-- must match your earlier STUDY_NAME

study = optuna.load_study(study_name=STUDY_NAME, storage=STUDY_DB)
print("Trials:", len(study.trials), "Best:", study.best_value)

# --- 1) Optimization history ---
fig = plot_optimization_history(study)
fig.show()

# --- 2) Parameter importance (most important params) ---
fig = plot_param_importances(study)  # uses fANOVA when possible
fig.show()

# --- Optional: helpful diagnostics ---
# Slice plot: one subplot per parameter vs objective
fig = plot_slice(study)
fig.show()

# Parallel coordinate: interactions across many params
fig = plot_parallel_coordinate(study)
fig.show()

In [ ]:

print("\nTensorBoard:")
print(f"  In Colab, run: %load_ext tensorboard; %tensorboard --logdir {TB_ROOT}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/'Colab Notebooks'

/content/drive/MyDrive/Colab Notebooks
